# Word Embeddings

In this demonstration, we use [spaCy](https://spacy.io/)'s pre-trained word embeddings, then learn our own on a corpus of forum posts, with a skip-gram written in PyTorch.

In [1]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.9 MB/s  0:01:19
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


In [2]:
import collections
import itertools
import pathlib
import re
import time

import numpy as np
import spacy
import torch

## spaCy's vectors

`en_core_web_lg`, spaCy's large English model, maps 684,830 words to 342,918 vectors of 300 dimensions: the case variants of a word share one vector. Each word of an analyzed text has its vector, and the whole text the mean of them.

In [3]:
nlp = spacy.load("en_core_web_lg")
doc = nlp("This is some text that I am processing with spaCy")

print(doc[3].text, doc[3].vector.shape)
print(doc[3].vector[:10])

print("Mean vector of the sentence:", doc.vector.shape)
print(doc.vector[:10])

text (300,)
[ 0.037103 -0.31259  -0.17857   0.30001   0.078154  0.17958   0.12048
 -0.11879  -0.20601   1.2849  ]
Mean vector of the sentence: (300,)
[-0.0807013   0.2231553  -0.13447897 -0.0164909   0.0356182   0.1719778
  0.06692369 -0.2001477  -0.03886471  1.8532441 ]


## An embedding table

The `Embeddings` class stores the vectors in a table, one per row, scaled to length 1: the dot product of two rows is then their cosine. It will also hold the vectors we train.

In [4]:
class Embeddings:
  """An embedding table: one vector of length 1 per word."""

  def __init__(self, rows, table):
    self.rows = rows  # word -> number of its row in the table
    self.words = {}  # row number -> word (in lowercase if there are several)
    for word, row in rows.items():
      if row not in self.words or word.islower():
        self.words[row] = word
    # Rows of length 1: the dot product of two rows is their cosine
    self.table = table / np.linalg.norm(table, axis=1, keepdims=True)

  def __contains__(self, word):
    return word in self.rows

  def __getitem__(self, word):
    return self.table[self.rows[word]]

  def neighbors(self, vector, n=5):
    """The n words closest to `vector`: (word, cosine) pairs."""
    cosine = self.table @ (vector / np.linalg.norm(vector))
    return [
      (self.words[i], round(float(cosine[i]), 2)) for i in np.argsort(-cosine)[:n]
    ]

In [5]:
vectors = nlp.vocab.vectors
model = Embeddings(
  {nlp.vocab.strings[key]: row for key, row in vectors.key2row.items()},
  vectors.data,
)
print(len(model.rows), "words,", model.table.shape)

684830 words, (342918, 300)


### Known and unknown words

`Berlin` and `berlin` are both known, and share one vector; a misspelled word has none. A text is preprocessed like the ones the vectors were learned from.

In [6]:
for word in ["Berlin", "berlin", "Windoows"]:
  print(word, word in model)

Berlin True
berlin True
Windoows False


### Neighbors

A word's neighbors are the rows with the highest cosine with its own. `dos` and `DOS` share a vector too, the Spanish word's.

In [7]:
for word in ["Berlin", "berlin", "dos", "DOS"]:
  print(word, model.neighbors(model[word]))

Berlin [('berlin', 1.0), ('hamburg', 0.75), ('germany', 0.74), ('frankfurt', 0.73), ('munich', 0.71)]
berlin [('berlin', 1.0), ('hamburg', 0.75), ('germany', 0.74), ('frankfurt', 0.73), ('munich', 0.71)]
dos [('dos', 1.0), ('os', 0.67), ('todos', 0.56), ('que', 0.56), ('para', 0.55)]
DOS [('dos', 1.0), ('os', 0.67), ('todos', 0.56), ('que', 0.56), ('para', 0.55)]


### Similarity of two words

The dot product of two vectors of length 1 is their cosine.

In [8]:
print(model["Berlin"] @ model["Munich"])
print(model["Berlin"] @ model["Paris"])

0.7140046
0.5989031


## Analogies

`x` is to `y` what `z` is to? We look for the neighbors of `y - x + z`. *France* is to *Paris* what *Japan* is to:

In [9]:
model.neighbors(model["Paris"] - model["France"] + model["Japan"])

[('japan', 0.84),
 ('tokyo', 0.8),
 ('japanese', 0.74),
 ('paris', 0.64),
 ('asian', 0.63)]

First comes `japan`, a word of the question (`Japan` and `japan` share a vector): `Paris - France` barely moves the vector of `Japan`. The `analogy` function therefore leaves out the words of the question.

In [10]:
def analogy(model, x, y, z):
  # x is to y what z is to? The words of the question are left out
  vector = model[y] - model[x] + model[z]
  # The rows of the words of the question (Japan and japan share theirs)
  question = {model.rows[word] for word in (x, y, z)}
  nearest = model.neighbors(vector, n=8)
  return [(word, c) for word, c in nearest if model.rows[word] not in question][:5]


analogy(model, "France", "Paris", "Japan")

[('tokyo', 0.8),
 ('japanese', 0.74),
 ('asian', 0.63),
 ('osaka', 0.6),
 ('asia', 0.6)]

### A success, a failure, a bias

*Man* is to *woman* what *king* is to:

In [11]:
analogy(model, "man", "woman", "king")

[('queen', 0.78),
 ('prince', 0.61),
 ('princess', 0.6),
 ('kings', 0.6),
 ('queens', 0.57)]

With Australia, Canberra isn't among the first five: an analogy is only a neighborhood, and better known Australian cities are closer.

In [12]:
analogy(model, "France", "Paris", "Australia")

[('sydney', 0.76),
 ('melbourne', 0.73),
 ('perth', 0.7),
 ('brisbane', 0.69),
 ('australian', 0.68)]

The vectors carry the associations of the texts they come from, stereotypes included.

In [13]:
analogy(model, "man", "woman", "doctor")

[('nurse', 0.69),
 ('doctors', 0.67),
 ('physician', 0.66),
 ('pregnant', 0.65),
 ('gynecologist', 0.63)]

## Training our own embeddings

### The 20 Newsgroups corpus

About 18,800 forum posts in English, spread over 20 topics.

In [14]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
!tar xzf 20news-bydate.tar.gz
!ls 20news-bydate-train/

--2026-11-05 18:40:03--  https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/shuuchuu/datasets/refs/heads/main/20newsgroups/20news-bydate.tar.gz [following]
--2026-11-05 18:40:04--  https://raw.githubusercontent.com/shuuchuu/datasets/refs/heads/main/20newsgroups/20news-bydate.tar.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14464277 (14M) [application/octet-stream]
Saving to: ‘20news-bydate.tar.gz’

20news-bydate.tar.g 100%[===================>]  13.79M  9.55MB/s    in 1.4s    

2026-11-05 18:40:05 (9.55

In [15]:
!ls 20news-bydate-test/

alt.atheism		  rec.autos	      sci.space
comp.graphics		  rec.motorcycles     soc.religion.christian
comp.os.ms-windows.misc   rec.sport.baseball  talk.politics.guns
comp.sys.ibm.pc.hardware  rec.sport.hockey    talk.politics.mideast
comp.sys.mac.hardware	  sci.crypt	      talk.politics.misc
comp.windows.x		  sci.electronics     talk.religion.misc
misc.forsale		  sci.med


In [16]:
contents = [
  p.read_text(encoding="latin-1") for p in pathlib.Path(".").glob("20news-bydate-*/*/*")
]
print(f"{len(contents)} texts retrieved")

18846 texts retrieved


In [17]:
print(contents[1])

From: jagst18+@pitt.edu (Josh A Grossman)
Subject: Re: Dillon puts foot in mouth, film at 11
Organization: The Zets at the University of Pittsburgh (aka The Pitt)
Lines: 17


I have also heard about HCI claiming thant anyone they  get an address
from is a member.  If this is the case their membership rolls are
grossly inflated and we should not call them and give them a name
and address to add to their already false rolls.  Perhaps
if you could get a copy of their existing membership, then pretend to
be an existing member, do that several thousand times, you could
hurt HCI.  But names are power.  Remeber the NRA uses the fact
that it has 3 million paid members in order to flex its muscles.

Perhaps politicians don't realize the lying tactics of HCI, wait a
minute, HCI learned it from politicians....

Later,
Josh





`preprocess` removes a post's header and the lines announcing a quote ("… writes:"), then splits the rest into words, on spaces and punctuation. The case stays.

In [18]:
def preprocess(content: str) -> list[str]:
  start = content.find("\n\n")
  if start > 0:
    content = content[start + 2 :]
  words_per_line = (
    re.sub(r'[\!"#$%&\*+,-./:;<=>?@^_`()|~=]', " ", line).split()
    for line in content.splitlines()
    if not line.endswith("writes:")
  )
  return list(itertools.chain.from_iterable(words_per_line))


texts = [preprocess(content) for content in contents]
print(texts[0])

['Do', 'you', 'know', 'how', 'many', 'deaths', 'each', 'year', 'are', 'caused', 'by', 'self', 'inflicted', 'gun', 'shot', 'wounds', 'by', 'people', 'wearing', 'thigh', 'holsters', 'There', 'are', 'roughly', '1200', 'fatal', 'firearms', 'related', 'accidents', 'each', 'year', 'The', 'large', 'majority', 'involve', 'rifles', 'and', 'shotgun', 'there', 'are', 'under', '500', 'fatal', 'handgun', 'accidents', 'each', 'year', 'I', 'really', 'doubt', 'all', 'of', 'those', 'occur', 'while', 'the', 'pistol', 'is', 'holstered', 'so', 'the', 'number', 'of', 'self', 'inflicted', 'gunshot', 'wounds', 'by', 'people', 'wearing', 'thigh', 'holsters', 'is', 'probably', 'well', 'under', '250', 'per', 'year', "I'm", 'neither', 'a', 'doctor', 'nor', 'a', 'firearms', 'tech', 'expert', 'but', 'it', 'would', 'seem', 'that', 'given', 'the', 'way', 'a', 'holstered', 'gun', 'points', 'accidental', 'injuries', 'inflicted', 'that', 'way', 'would', 'be', 'among', 'the', 'least', 'lethal', 'David', 'Veal', 'Univ', 

### Frequent phrases

*New York* names a single thing: one word `New_York` is better than two. Two words that follow each other much more often than their frequencies would suggest form a phrase, following the score of [Mikolov et al. (2013)](https://arxiv.org/abs/1310.4546): $(\text{count}(ab) - \delta) / (\text{count}(a) \times \text{count}(b))$. It takes a few seconds.

In [19]:
# Two words that follow each other much more often than chance would have it form
# a phrase: score of Mikolov et al. (2013), (count(ab) - δ) / (count(a) × count(b))
word_counts = collections.Counter(word for text in texts for word in text)
pair_counts = collections.Counter(p for text in texts for p in zip(text, text[1:]))
total = sum(word_counts.values())  # × total: the score becomes a ratio to chance
phrases = {
  (a, b)
  for (a, b), count in pair_counts.items()
  if (count - 5) * total / (word_counts[a] * word_counts[b]) > 100
}


def group(text):
  # Replace each pair of a phrase with a single word, a_b
  grouped = []
  for word in text:
    if grouped and (grouped[-1], word) in phrases:
      grouped[-1] = grouped[-1] + "_" + word
    else:
      grouped.append(word)
  return grouped


grouped_texts = [group(text) for text in texts]

In [20]:
print(len(phrases), "phrases")
print(sorted(a + "_" + b for a, b in phrases if a == "New"))
print(grouped_texts[0][:20])

20429 phrases
['New_Age', 'New_Brunswick', 'New_England', 'New_Hampshire', 'New_Haven', 'New_Hobgoblin', 'New_Jersey', 'New_Mexico', 'New_Mutants', 'New_Orleans', 'New_Regime', 'New_Testament', 'New_Warriors', 'New_York', 'New_Yorker', 'New_Zealand']
['Do', 'you', 'know', 'how', 'many', 'deaths', 'each', 'year', 'are', 'caused_by', 'self', 'inflicted', 'gun', 'shot', 'wounds', 'by', 'people', 'wearing', 'thigh', 'holsters']


### A skip-gram in PyTorch

These embeddings, called static, used to be trained with a dedicated library, such as gensim or fastText: that is the historical way. Today, representations are learned inside the models (transformers), and [gensim](https://github.com/piskvorky/gensim) is in maintenance mode, with no new features. A skip-gram with negative sampling fits in a few lines of PyTorch.

The vocabulary keeps the words seen at least 5 times, and the corpus becomes a single sequence of word numbers. A very frequent word is removed most of the time: it teaches its neighbors nothing more.

In [21]:
DIMENSION, WINDOW, NEGATIVES, BATCH = 100, 5, 5, 4096
EPOCHS = 3

# Vocabulary: the words seen at least 5 times, from the most frequent to the rarest
counts = collections.Counter(word for text in grouped_texts for word in text)
vocabulary = [word for word, count in counts.most_common() if count >= 5]
numbers = {word: number for number, word in enumerate(vocabulary)}
frequencies = torch.tensor([counts[word] for word in vocabulary], dtype=torch.float)
# The corpus: a single sequence of word numbers
corpus = torch.tensor(
  [numbers[word] for text in grouped_texts for word in text if word in numbers]
)
# Subsampling: a very frequent word (the, of…) is removed most of the time
torch.manual_seed(0)
keep = torch.rand(len(corpus)) < torch.sqrt(
  1e-3 * frequencies.sum() / frequencies[corpus]
)
corpus = corpus[keep]
print(len(vocabulary), "words,", len(corpus), "occurrences kept")

57189 words, 3501077 occurrences kept


The model has two tables: the vector of a word at the center of the window, and its vector when it is a neighbor. The score of a pair is the dot product of their two vectors.

In [22]:
class SkipGram(torch.nn.Module):
  def __init__(self, vocabulary_size, dimension):
    super().__init__()
    # Two tables: the vector of a word at the center, and its vector as a neighbor
    self.centers = torch.nn.Embedding(vocabulary_size, dimension, sparse=True)
    self.contexts = torch.nn.Embedding(vocabulary_size, dimension, sparse=True)
    torch.nn.init.uniform_(self.centers.weight, -0.5 / dimension, 0.5 / dimension)
    torch.nn.init.zeros_(self.contexts.weight)

  def forward(self, center, context):
    # Score of a pair: the dot product of its two vectors
    return (self.centers(center) * self.contexts(context)).sum(dim=-1)

The loss pushes the score of a true neighbor up, and that of words drawn at random, the negatives, down.

In [23]:
def loss(skipgram, center, context, negatives):
  # -log σ(true neighbor's score) - Σ log σ(-a negative's score), mean over the batch
  real = torch.nn.functional.logsigmoid(skipgram(center, context))
  noise = torch.nn.functional.logsigmoid(-skipgram(center[:, None], negatives)).sum(
    dim=1
  )
  return -(real + noise).mean()

Training: each batch draws positions in the corpus, a neighbor in the window of each, and negatives. An epoch takes about a minute on two processors, like Colab's.

In [24]:
skipgram = SkipGram(len(vocabulary), DIMENSION)
optimizer = torch.optim.SparseAdam(skipgram.parameters(), lr=0.003)
negative_weights = frequencies**0.75  # a little more chance for rare words

start = time.time()
for epoch in range(EPOCHS):
  total = 0.0
  batches = torch.randperm(len(corpus) - 2 * WINDOW).add(WINDOW).split(BATCH)
  for positions in batches:
    # A neighbor drawn in the window: 1..WINDOW words away, on the left or the right
    offset = torch.randint(1, WINDOW + 1, positions.shape)
    offset = offset * (2 * torch.randint(0, 2, positions.shape) - 1)
    center, context = corpus[positions], corpus[positions + offset]
    negatives = torch.multinomial(
      negative_weights, len(positions) * NEGATIVES, replacement=True
    )
    value = loss(skipgram, center, context, negatives.view(-1, NEGATIVES))
    optimizer.zero_grad()
    value.backward()
    optimizer.step()
    total += value.item()
  print(
    f"epoch {epoch + 1}: loss {total / len(batches):.3f}, {time.time() - start:.0f} s"
  )

epoch 1: loss 2.820, 51 s
epoch 2: loss 2.445, 125 s
epoch 3: loss 2.368, 202 s


### The learned vectors

The table of centers goes into the same class as spaCy's vectors. On this corpus of the domain, the neighbors of `Windows` and `DOS` come from the computing forums; `New_York`, a word of our vocabulary, is missing from spaCy's, whose tokenizer splits *New York* in two.

In [25]:
model_forums = Embeddings(numbers, skipgram.centers.weight.detach().numpy())

for word in ["Windows", "DOS", "New_York"]:
  print(word, model_forums.neighbors(model_forums[word]))
  if word in model:
    print("  spaCy:", model.neighbors(model[word]))
  else:
    print("  spaCy: missing")

Windows [('Windows', 1.0), ('Windows_NT', 0.87), ('Excel', 0.87), ('OS_2', 0.86), ('windows', 0.86)]
  spaCy: [('windows', 1.0), ('window', 0.75), ('xp', 0.7), ('vista', 0.7), ('microsoft', 0.61)]
DOS [('DOS', 1.0), ('MS_DOS', 0.84), ('MS_Windows', 0.84), ('tcp_ip', 0.83), ('Windows_NT', 0.82)]
  spaCy: [('dos', 1.0), ('os', 0.67), ('todos', 0.56), ('que', 0.56), ('para', 0.55)]
New_York [('New_York', 1.0), ('replaced_Jimmy', 0.86), ('League', 0.85), ('1982', 0.85), ('Carter', 0.84)]
  spaCy: missing


Five million words aren't enough for analogies:

In [26]:
analogy(model_forums, "man", "woman", "king")

[('Women', 0.87),
 ('west', 0.86),
 ('rose', 0.84),
 ('voted', 0.84),
 ('elite', 0.83)]